# Niche Analysis & Selection
**Goal:** Data-driven niche selection for maximum revenue potential.

This notebook analyzes niches across multiple dimensions:
- Revenue per 1,000 views (RPM)
- Competition level
- AI content suitability
- Trend direction
- Affiliate opportunity score
- Composite weighted score

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

## 1. Niche Data (2026 Research)

In [ ]:
# Comprehensive niche data from research
niches = pd.DataFrame({
    'Niche': [
        'AI & Technology', 'Personal Finance', 'Business & Entrepreneurship',
        'True Crime & Mystery', 'Health & Wellness', 'Education & How-To',
        'Sleep & Relaxation', 'Animated Stories', 'Crypto & Web3',
        'Productivity & Tools', 'Science & Space', 'History & Documentaries',
        'Gaming News', 'Psychology & Self-Help', 'Real Estate'
    ],
    'RPM_Low': [12, 15, 10, 8, 8, 9, 4, 9, 12, 10, 7, 6, 5, 9, 14],
    'RPM_High': [30, 40, 25, 13, 15, 14, 8, 13, 35, 20, 14, 11, 10, 16, 30],
    'Competition': [3, 4, 3, 3, 3, 3, 1, 3, 4, 2, 2, 2, 4, 3, 4],  # 1=low, 5=extreme
    'AI_Suitability': [5, 5, 5, 3, 3, 5, 5, 3, 4, 5, 4, 4, 3, 4, 4],  # 1-5
    'Trend': [5, 3, 3, 3, 4, 3, 4, 3, 2, 4, 3, 3, 3, 4, 3],  # 1=declining, 5=rapidly rising
    'Affiliate_Score': [5, 5, 4, 2, 4, 4, 2, 1, 4, 5, 2, 2, 3, 3, 5],  # 1-5
    'Evergreen_Score': [3, 5, 4, 5, 5, 5, 5, 4, 2, 4, 5, 5, 2, 5, 4],  # 1-5 (how long content stays relevant)
    'Audience_Size': [4, 5, 4, 4, 5, 5, 3, 3, 3, 4, 3, 3, 5, 4, 3],  # 1-5
})

# Calculate derived metrics
niches['RPM_Avg'] = (niches['RPM_Low'] + niches['RPM_High']) / 2
niches['RPM_Range'] = niches['RPM_High'] - niches['RPM_Low']

# Monthly revenue estimate (assuming 500K views/month at maturity)
niches['Est_Monthly_Rev_Low'] = (niches['RPM_Low'] * 500)
niches['Est_Monthly_Rev_High'] = (niches['RPM_High'] * 500)

niches.sort_values('RPM_Avg', ascending=False)

## 2. Composite Scoring Model
Weighted scoring to find the optimal niche. Weights reflect our priorities:
- Revenue (RPM): 30% — primary income driver
- AI Suitability: 20% — automation is key to our strategy
- Low Competition: 15% — easier to break through
- Trend: 15% — growing niches = growing revenue
- Affiliate Potential: 10% — secondary income
- Evergreen Content: 10% — passive income longevity

In [ ]:
# Normalize all scores to 0-1 range
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

# Weights (must sum to 1.0)
WEIGHTS = {
    'RPM': 0.30,
    'AI_Suitability': 0.20,
    'Low_Competition': 0.15,
    'Trend': 0.15,
    'Affiliate': 0.10,
    'Evergreen': 0.10,
}

# Normalize and invert competition (lower = better)
niches['n_RPM'] = normalize(niches['RPM_Avg'])
niches['n_AI'] = normalize(niches['AI_Suitability'])
niches['n_Competition'] = normalize(5 - niches['Competition'])  # Inverted
niches['n_Trend'] = normalize(niches['Trend'])
niches['n_Affiliate'] = normalize(niches['Affiliate_Score'])
niches['n_Evergreen'] = normalize(niches['Evergreen_Score'])

# Composite score
niches['Composite_Score'] = (
    niches['n_RPM'] * WEIGHTS['RPM'] +
    niches['n_AI'] * WEIGHTS['AI_Suitability'] +
    niches['n_Competition'] * WEIGHTS['Low_Competition'] +
    niches['n_Trend'] * WEIGHTS['Trend'] +
    niches['n_Affiliate'] * WEIGHTS['Affiliate'] +
    niches['n_Evergreen'] * WEIGHTS['Evergreen']
)

# Rank
niches['Rank'] = niches['Composite_Score'].rank(ascending=False).astype(int)
results = niches[['Niche', 'RPM_Avg', 'Competition', 'AI_Suitability', 'Trend', 
                   'Affiliate_Score', 'Evergreen_Score', 'Composite_Score', 'Rank']].sort_values('Rank')

print('=' * 80)
print('NICHE RANKING BY COMPOSITE SCORE')
print('=' * 80)
results.to_string(index=False)

In [ ]:
# Visualization: Composite Score Bar Chart
fig, ax = plt.subplots(figsize=(14, 8))
ranked = niches.sort_values('Composite_Score', ascending=True)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(ranked)))

bars = ax.barh(ranked['Niche'], ranked['Composite_Score'], color=colors, edgecolor='white', linewidth=0.5)

# Add value labels
for bar, score in zip(bars, ranked['Composite_Score']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
            f'{score:.2f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Composite Score', fontsize=13)
ax.set_title('Niche Ranking: Weighted Composite Score\n(RPM 30% | AI Suitability 20% | Competition 15% | Trend 15% | Affiliate 10% | Evergreen 10%)', 
             fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.1)
plt.tight_layout()
plt.savefig('../output/niche_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/niche_ranking.png')

In [ ]:
# Visualization: RPM Revenue Potential
fig, ax = plt.subplots(figsize=(14, 8))
ranked = niches.sort_values('RPM_Avg', ascending=True)

ax.barh(ranked['Niche'], ranked['RPM_Low'], color='#2E75B6', alpha=0.4, label='RPM Low')
ax.barh(ranked['Niche'], ranked['RPM_High'] - ranked['RPM_Low'], 
        left=ranked['RPM_Low'], color='#2E75B6', alpha=0.8, label='RPM Range (to High)')

for i, row in ranked.iterrows():
    ax.text(row['RPM_High'] + 0.5, row['Niche'], f"${row['RPM_Low']}-${row['RPM_High']}", 
            va='center', fontsize=9)

ax.set_xlabel('Revenue Per 1,000 Views ($)', fontsize=13)
ax.set_title('YouTube RPM by Niche (2026 Data)\nHigher = More Revenue Per View', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../output/rpm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/rpm_comparison.png')

In [ ]:
# Visualization: Revenue Projection (500K monthly views)
fig, ax = plt.subplots(figsize=(14, 8))
ranked = niches.sort_values('Est_Monthly_Rev_High', ascending=True)

ax.barh(ranked['Niche'], ranked['Est_Monthly_Rev_Low'], color='#4CAF50', alpha=0.4, label='Conservative')
ax.barh(ranked['Niche'], ranked['Est_Monthly_Rev_High'] - ranked['Est_Monthly_Rev_Low'],
        left=ranked['Est_Monthly_Rev_Low'], color='#4CAF50', alpha=0.8, label='Upside')

# Add $5K target line
ax.axvline(x=5000, color='red', linestyle='--', linewidth=2, label='$5,000/mo Target')

for i, row in ranked.iterrows():
    ax.text(row['Est_Monthly_Rev_High'] + 100, row['Niche'], 
            f"${row['Est_Monthly_Rev_Low']:,.0f}-${row['Est_Monthly_Rev_High']:,.0f}", 
            va='center', fontsize=9)

ax.set_xlabel('Estimated Monthly AdSense Revenue ($)', fontsize=13)
ax.set_title('Monthly Revenue Potential at 500K Views/Month (AdSense Only)\nRed line = $5,000 Target', 
             fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../output/revenue_projection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/revenue_projection.png')

In [ ]:
# Radar Chart: Top 5 niches comparison
top5 = niches.sort_values('Composite_Score', ascending=False).head(5)
categories = ['RPM', 'AI Suitability', 'Low Competition', 'Trend', 'Affiliate', 'Evergreen']
norm_cols = ['n_RPM', 'n_AI', 'n_Competition', 'n_Trend', 'n_Affiliate', 'n_Evergreen']

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
colors_radar = ['#2E75B6', '#E74C3C', '#2ECC71', '#F39C12', '#9B59B6']

for idx, (_, row) in enumerate(top5.iterrows()):
    values = [row[c] for c in norm_cols]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Niche'], color=colors_radar[idx])
    ax.fill(angles, values, alpha=0.1, color=colors_radar[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_title('Top 5 Niches: Multi-Dimension Comparison', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('../output/niche_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/niche_radar.png')

## 3. Google Trends Integration (Live Data)
Uncomment and run to pull live Google Trends data for your candidate niches.

In [ ]:
# ─── Google Trends (uncomment to use) ────────────────────────
# from pytrends.request import TrendReq
#
# pytrends = TrendReq(hl='en-US', tz=360)
# keywords = ['AI tools', 'personal finance tips', 'productivity apps', 'crypto news', 'sleep sounds']
# pytrends.build_payload(keywords, cat=0, timeframe='today 12-m')
# trends_df = pytrends.interest_over_time()
#
# if not trends_df.empty:
#     trends_df.drop('isPartial', axis=1, errors='ignore').plot(figsize=(14, 6))
#     plt.title('Google Trends: 12-Month Interest', fontsize=14, fontweight='bold')
#     plt.ylabel('Search Interest (relative)')
#     plt.tight_layout()
#     plt.savefig('../output/google_trends.png', dpi=150)
#     plt.show()

print('Uncomment the Google Trends code above to pull live trend data.')
print('This requires: pip install pytrends')

## 4. Final Recommendation

In [ ]:
top3 = niches.sort_values('Composite_Score', ascending=False).head(3)

print('=' * 70)
print('FINAL NICHE RECOMMENDATIONS')
print('=' * 70)

for rank, (_, row) in enumerate(top3.iterrows(), 1):
    print(f"\n{'#' + str(rank)} {row['Niche']}")
    print(f"   Composite Score: {row['Composite_Score']:.3f}")
    print(f"   RPM Range: ${row['RPM_Low']}-${row['RPM_High']} per 1K views")
    print(f"   Monthly Potential (500K views): ${row['Est_Monthly_Rev_Low']:,.0f}-${row['Est_Monthly_Rev_High']:,.0f}")
    print(f"   AI Suitability: {'★' * row['AI_Suitability']}{'☆' * (5-row['AI_Suitability'])}")
    print(f"   Competition: {'●' * row['Competition']}{'○' * (5-row['Competition'])}")

print('\n' + '=' * 70)
print('RECOMMENDED STRATEGY:')
print(f"  Primary Channel:   {top3.iloc[0]["Niche"]}")
print(f"  Secondary Channel: {top3.iloc[1]["Niche"]} (launch at month 3-4)")
print(f"  Tertiary Channel:  {top3.iloc[2]["Niche"]} (launch at month 6+)")
print('=' * 70)